In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="distilbert_no_special_tokens_mean_cosine_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [3]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print("cls_token_id:", tokenizer.cls_token_id)
print("sep_token_id:", tokenizer.sep_token_id)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
cls_token_id: 101
sep_token_id: 102


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
def mean_pool_excluding_special(last_hidden_state, attention_mask, input_ids):
    mask = attention_mask.clone().bool()
    if tokenizer.cls_token_id is not None:
        mask &= input_ids != tokenizer.cls_token_id
    if tokenizer.sep_token_id is not None:
        mask &= input_ids != tokenizer.sep_token_id

    mask_f = mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask_f).sum(dim=1)
    counts = mask_f.sum(dim=1).clamp(min=1.0)
    pooled = summed / counts
    pooled = F.normalize(pooled, p=2, dim=-1, eps=1e-12)
    return pooled

def encode_texts(texts, batch_size=128, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = model(**enc)
            pooled = mean_pool_excluding_special(
                outputs.last_hidden_state,
                enc["attention_mask"],
                enc["input_ids"],
            )
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


In [6]:
emb1 = encode_texts(sent1, batch_size=128, max_length=128)
print("emb1 shape:", emb1.shape)


  0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 768)


In [7]:
emb2 = encode_texts(sent2, batch_size=128, max_length=128)
print("emb2 shape:", emb2.shape)


  0%|          | 0/4 [00:00<?, ?it/s]

emb2 shape: (408, 768)


In [8]:
cosine_scores = (emb1 * emb2).sum(axis=1)
threshold = 0.85
y_pred = (cosine_scores >= threshold).astype(int)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1, "threshold": threshold})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.7034313725490197, 'f1': 0.8212703101920237, 'threshold': 0.85}
                precision    recall  f1-score   support

not_paraphrase       0.90      0.07      0.13       129
    paraphrase       0.70      1.00      0.82       279

      accuracy                           0.70       408
     macro avg       0.80      0.53      0.48       408
  weighted avg       0.76      0.70      0.60       408



In [ ]:

vault.create_record_list("no_special_cosine_mrpc_similarity_prediction", column_names=["prediction", "cosine_scores"])

for i in range(len(y_pred)):
    vault.append_record("no_special_cosine_mrpc_similarity_prediction", 
                        {
                            "prediction": y_pred[i],
                            "cosine_scores": float(cosine_scores[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT no_special_cosine_mrpc_similarity_prediction"
embedding = get_embeddings(description)
vault.create_description("no_special_cosine_mrpc_similarity_prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("no_special_cosine_mrpc_similarity_prediction", cat, embedding, prop)

In [9]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(cosine_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9537599086761475
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.895621657371521
true: 0 pred: 1
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.9625198841094971
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will deci

In [10]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(cosine_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


num_errors: 121
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.895621657371521
true: 0 pred: 1
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.9625198841094971
true: 0 pred: 1
idx: 4
sentence1: No dates have been set for the civil or the criminal trial .
sentence2: No dates have been set for the criminal or civil cases , but Shanley has pleaded not guilty .
score: 0.9389275908470154
true: 0 pred: 1
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , s

In [11]:
vault.create_record_list("distilbert_no_special_tokens_mean_cosine_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("distilbert_no_special_tokens_mean_cosine_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "no_special_cosine_mrpc_similarity_prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT distilbert_no_special_tokens_mean_cosine_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("distilbert_no_special_tokens_mean_cosine_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_no_special_tokens_mean_cosine_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'distilbert-base-uncased',
 'device': 'mps',
 'pooling': 'mean_excluding_special_tokens',
 'threshold': 0.85,
 'num_examples': 408,
 'accuracy': 0.7034313725490197,
 'f1': 0.8212703101920237}

In [ ]:
description = "INSERT TEXT HERE ABOUT distilbert_no_special_tokens_mean_cosine_mrpc process/notebook" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("distilbert_no_special_tokens_mean_cosine_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_no_special_tokens_mean_cosine_mrpc", cat, embedding, prop)